In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score


In [2]:
# Generate a synthetic dataset: 5000 rows, 20 columns
rng = np.random.RandomState(42)
n_rows = 5000

numeric_cols = [f"num_{i}" for i in range(1, 16)]   # 15 numeric features
cat_cols = [f"cat_{i}" for i in range(1, 6)]         # 5 categorical features

data = {}
for col in numeric_cols:
    data[col] = rng.normal(loc=rng.uniform(10, 100), scale=rng.uniform(5, 25), size=n_rows)

for col in cat_cols:
    n_levels = rng.randint(3, 6)
    levels = [f"level_{j}" for j in range(n_levels)]
    data[col] = rng.choice(levels, size=n_rows)

df = pd.DataFrame(data)

# Inject missing values into a few numeric columns
for col in rng.choice(numeric_cols, size=4, replace=False):
    missing_idx = rng.choice(df.index, size=int(0.05 * n_rows), replace=False)
    df.loc[missing_idx, col] = np.nan

# Build a continuous target as a weighted combination of numeric features + noise
weights = rng.uniform(-3, 3, size=len(numeric_cols))
target = df[numeric_cols].fillna(df[numeric_cols].mean()).values @ weights
target += rng.normal(0, 15, size=n_rows)
df["target"] = target

df.shape


(5000, 21)

In [3]:
X = df.drop(columns=["target"])
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


In [4]:
preprocessor = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]), numeric_cols),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ]), cat_cols)
])

pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", Ridge(alpha=1.0))
])

pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)


In [5]:
mse = mean_squared_error(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"MSE: {mse:.3f}")
print(f"MAE: {mae:.3f}")
print(f"R2:  {r2:.3f}")


MSE: 231.806
MAE: 12.042
R2:  0.954


In [6]:
cv_scores = cross_val_score(pipeline, X_train, y_train, cv=5, scoring="neg_mean_absolute_error")
print("CV MAE:", -cv_scores.mean())


CV MAE: 12.159004451178031


In [7]:
param_grid = {"model__alpha": [0.01, 0.1, 1.0, 10.0, 100.0]}

grid_search = GridSearchCV(pipeline, param_grid, cv=5, scoring="neg_mean_absolute_error")
grid_search.fit(X_train, y_train)

print("Best alpha:", grid_search.best_params_)
print("Best CV MAE:", -grid_search.best_score_)


Best alpha: {'model__alpha': 1.0}
Best CV MAE: 12.159004451178031


In [8]:
best_model = grid_search.best_estimator_
y_pred_best = best_model.predict(X_test)

print("Baseline Ridge -> MAE: {:.3f}, R2: {:.3f}".format(mae, r2))
print("Tuned Ridge    -> MAE: {:.3f}, R2: {:.3f}".format(
    mean_absolute_error(y_test, y_pred_best),
    r2_score(y_test, y_pred_best)
))


Baseline Ridge -> MAE: 12.042, R2: 0.954
Tuned Ridge    -> MAE: 12.042, R2: 0.954
